# SAE Feature Explainer For One Synthetic-Lethality Pair

This notebook projects one SLformer pair into SAE dictionary atoms and explains selected atoms from activation exemplars.


In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml


PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "SAE").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from SAE.SAE_training.model import SAEConfig, SparseAutoencoder
from SAE.SAE_training.utils.data import build_artifacts, extract_concat_matrix
from SAE.LLM_pipeline.utils.explainer import (
    build_explainer_prompt,
    extract_explainer_fields,
    rank_candidate_features,
    activation_feature_tables,
    target_relevance_note,
)
from SAE.LLM_pipeline.utils.general import find_pair_index, sample_name, write_json
from SAE.LLM_pipeline.utils.llm_strategy import final_strategy_text, run_llm_strategy, strategy_call_count, strategy_total_tokens
from SAE.manifold.utils.projection import candidate_feature_table, encode_dataset, projection_state
from prompt_api.client import AigcBestChatClient

print("Project root:", PROJECT_ROOT)

## Configuration

The notebook-level JSON is the single input for target selection and experiment controls. SAE training parameters come from the referenced training YAML, manifold parameters from the referenced manifold YAML, and client settings from the prompt API YAML and model settings from the SAE model YAML.

In [ ]:
NOTEBOOK_CONFIG = PROJECT_ROOT / "src" / "SAE" / "LLM_pipeline" / "config" / "explainer_simulator_config.yaml"
notebook_cfg = yaml.safe_load(NOTEBOOK_CONFIG.read_text(encoding="utf-8"))
train_cfg = yaml.safe_load((PROJECT_ROOT / notebook_cfg["train_config"]).read_text(encoding="utf-8"))
manifold_cfg = yaml.safe_load((PROJECT_ROOT / notebook_cfg["manifold_config"]).read_text(encoding="utf-8"))
prompt_api_config = PROJECT_ROOT / notebook_cfg["prompt_api_config"]

paths_cfg = train_cfg["paths"]
scope_cfg = train_cfg["scope"]
norm_cfg = train_cfg["normalization"]
model_cfg = SAEConfig(**dict(train_cfg["model"]))
projection_cfg = manifold_cfg["projection"]

TRAIN_SCOPE = str(scope_cfg["cancer"])
SEED = int(scope_cfg["seed"])
SCORE_COL = str(scope_cfg["score_col"])
EPS = float(norm_cfg["eps"])
TARGET_PRIMARY = str(notebook_cfg["target"]["primary_gene"])
TARGET_PARTNER = str(notebook_cfg["target"]["partner_gene"])
TARGET_CANCER = str(notebook_cfg["target"]["cancer"])
EXISTS_GROUNDTRUTH = bool(notebook_cfg["target"]["exists_groundtruth"])
strategy_cfg = notebook_cfg["llm_strategy"]
FEATURE_EXPLAINER_STRATEGY = str(strategy_cfg["feature_explainer"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

run_name = f"hidden{model_cfg.d_hidden}_gate{model_cfg.gate_weight}_orth{model_cfg.orth_weight}_k{model_cfg.topk}"
output_base_dir = Path(paths_cfg["output_base_dir"]).expanduser().resolve()
slformer_output_subdir = str(paths_cfg["slformer_output_subdir"])
run_dir = output_base_dir / slformer_output_subdir / TRAIN_SCOPE / run_name
output_dir = run_dir / "explanations" / f"{TARGET_CANCER}_{TARGET_PRIMARY}-{TARGET_PARTNER}"
output_dir.mkdir(parents=True, exist_ok=True)

print("Notebook config:", NOTEBOOK_CONFIG)
print("Run dir:", run_dir)
print("Output dir:", output_dir)
print("Training scope:", TRAIN_SCOPE)
print("Target cancer:", TARGET_CANCER)
print("Device:", DEVICE)

## Load Embeddings And SAE Coordinates

For within-cancer embeddings $X\in\mathbb{R}^{N\times d}$, apply the training normalization

$$\tilde X_{ij}=\frac{X_{ij}-\mu_j}{\sigma_j+\varepsilon}.$$

The trained SAE encoder maps each normalized pair to sparse coordinates $Z=f_\theta(\tilde X)\in\mathbb{R}^{N\times m}$.

In [ ]:
artifacts = build_artifacts(
    embeddings_pkl=Path(paths_cfg["embeddings_pkl"]).expanduser().resolve(),
    prediction_csvs=[Path(path).expanduser().resolve() for path in paths_cfg["prediction_csvs"]],
)
X, y, meta = extract_concat_matrix(artifacts, cancer=TRAIN_SCOPE, seed=SEED, use_score_col=SCORE_COL)

print("Base X:", X.shape)
print("Base rows:", len(meta))
print("Base cancer counts:")
display(meta["cancer"].value_counts().sort_index())


## Special IDH1–PRKDC Embedding Import

The IDH1–PRKDC Glioma target is a special outlier not present in the mix9 SAE artifact rows. This cell appends the extracted eight-context IDH1–PRKDC embeddings from `data/IDH1-PRKDC-emb` to the base mix manifold so downstream variables (`X`, `y`, `meta`, `target_idx`) remain aligned.


In [ ]:
SPECIAL_IDH1_DIR = PROJECT_ROOT / "data" / "IDH1-PRKDC-emb"
if (TARGET_PRIMARY, TARGET_PARTNER, TARGET_CANCER) == ("IDH1", "PRKDC", "Glioma"):
    idh1_X = np.load(SPECIAL_IDH1_DIR / "IDH1_PRKDC_context_embeddings.npy").astype(np.float32)
    idh1_meta = pd.read_csv(SPECIAL_IDH1_DIR / "IDH1_PRKDC_context_meta.csv")
    idh1_meta = idh1_meta[["primary_gene", "partner_gene", "cancer", "score"]].copy()
    idh1_meta["fold"] = "IDH1_PRKDC_special_best_fold_3"
    X = np.concatenate([X, idh1_X], axis=0).astype(np.float32, copy=False)
    y = np.concatenate([y, idh1_meta["score"].to_numpy(dtype=np.float32)], axis=0)
    meta = pd.concat([meta, idh1_meta], ignore_index=True)
    print("Appended special IDH1-PRKDC rows:", idh1_X.shape[0])
    print("Special source:", SPECIAL_IDH1_DIR)

target_idx = find_pair_index(meta, TARGET_PRIMARY, TARGET_PARTNER, TARGET_CANCER)
print("Aligned X:", X.shape)
print("Aligned rows:", len(meta))
print("Target index:", target_idx)
print("Target pair:", sample_name(meta, target_idx))
print("Target score:", float(y[target_idx]))


In [ ]:
mu = np.load(run_dir / "norm" / "mu.npy")
sigma = np.load(run_dir / "norm" / "sigma.npy")
Xn = ((X - mu) / (sigma + EPS)).astype(np.float32)

checkpoint = torch.load(run_dir / "final.pt", map_location="cpu")
model = SparseAutoencoder(SAEConfig(**checkpoint["sae_cfg"]))
model.load_state_dict(checkpoint["state_dict"])
model.eval().to(DEVICE)

Z = encode_dataset(model, Xn, batch_size=int(notebook_cfg["runtime"]["encode_batch_size"]), device=DEVICE)

print("Xn:", Xn.shape)
print("Z:", Z.shape)


## Local Manifold Projection State

Around the target point $x_0$, estimate a tangent basis $B\in\mathbb{R}^{d\times k}$ using local PCA and fit

$$s(x)\approx \beta_0+g^\top B^\top(x-\mu).$$

The local score-increasing unit direction is $\hat v=Bg/\lVert Bg\rVert_2$. The SAE response is measured by $\dot z=J_f(x_0)\hat v$, $\Delta_\varepsilon z=f_\theta(x_0+\varepsilon\hat v)-f_\theta(x_0)$, and

$$c^*=\arg\min_c\lVert W_d c-\hat v\rVert_2^2+\lambda\lVert c\rVert_1.$$

Reference scales for the printed diagnostics:
- `||grad_ambient||` is in SL-score units per one normalized-embedding unit; compare it with the local neighbor score standard deviation or with percentiles across target pairs.
- `normal_frac = ||x_perp||^2 / (||x_parallel||^2 + ||x_perp||^2)` measures how much of the target offset is outside the fitted tangent space; with larger neighborhoods this is expected to rise as the neighborhood stops being locally tangent.
- `decoder_nnz / d_hidden` and `decoder_nnz / topK` give the useful scale for decoder sparsity: near 1 atom is ideal, near `topK` is compact, and hundreds/thousands of atoms means the direction is distributed even if decoder cosine is high.


In [ ]:
state = projection_state(model, Xn, y, target_idx=target_idx, projection_config=projection_cfg, device=DEVICE)

z0 = state["z0"]
jvp = state["jvp"]
delta_z = state["delta_z"]
c_star = state["c_star"]
local = state["local"]
decoder_projection = state["decoder_projection"]

grad_norm = float(np.linalg.norm(local["grad_ambient"]))
x_parallel_norm = float(np.linalg.norm(state["x_tangent"]))
x_perp_norm = float(np.linalg.norm(state["x_normal"]))
normal_frac = x_perp_norm**2 / (x_parallel_norm**2 + x_perp_norm**2 + 1e-12)
tangent_frac = 1.0 - normal_frac
neighbor_scores = y[local["neighbor_indices"]]
neighbor_score_std = float(np.std(neighbor_scores))
neighbor_score_range = float(np.max(neighbor_scores) - np.min(neighbor_scores))
decoder_nnz = int(decoder_projection["nnz"])
decoder_nnz_frac = decoder_nnz / int(model_cfg.d_hidden)
decoder_nnz_topk_multiple = decoder_nnz / int(model_cfg.topk)

print("Neighbors:", int(local["neighbor_indices"].shape[0]))
print(f"||grad_ambient||: {grad_norm:.6f}  (local score std={neighbor_score_std:.6f}, local score range={neighbor_score_range:.6f})")
print(f"||x_parallel||: {x_parallel_norm:.6f}")
print(f"||x_perp||: {x_perp_norm:.6f}")
print(f"Tangent fraction: {tangent_frac:.3f} | Normal fraction: {normal_frac:.3f}  (0.0 ideal, >0.3 broad-neighborhood/tangent-warning)")
print(f"Decoder cosine: {float(decoder_projection['cosine']):.6f}  (1.0 best, 0.0 orthogonal)")
print(f"Decoder rel_err: {float(decoder_projection['rel_err']):.6f}  (0.0 best)")
print(f"Decoder nnz: {decoder_nnz} / {int(model_cfg.d_hidden)} atoms = {decoder_nnz_frac:.3f} of dictionary = {decoder_nnz_topk_multiple:.1f} x encoder topK({int(model_cfg.topk)})")
print("Decoder nnz guide: 1 single-atom; <=topK compact; <=4*topK usable mixture; >25% dictionary dense/distributed.")


## Candidate SAE Features

Candidate features are selected by

$$\mathcal{C}(x_0)=\operatorname{TopK}(z_0)\cup\operatorname{TopK}(|\dot z|)\cup\operatorname{TopK}(|c^*|).$$

This captures baseline activation, local differential sensitivity, and decoder participation.

In [ ]:
candidate_table = candidate_feature_table(
    z0,
    jvp,
    delta_z,
    c_star,
    topk=int(notebook_cfg["feature_selection"]["candidate_topk"]),
)

print("Candidate features:", len(candidate_table))
candidate_table.sort_values("jvp", key=lambda values: values.abs(), ascending=False).head(10)

## Activation Exemplars From Configured Evidence Pool

For each candidate feature $j$, collect high-activation and low-activation non-target pairs from the configured evidence pool. For a mix-trained SAE, `evidence.sample_scope: all_cancer` uses the global all-cancer manifold for feature labeling while keeping the analyzed target context fixed.


In [ ]:
%%time

evidence_table, explain_indices = activation_feature_tables(
    candidate_features=candidate_table["feature"].tolist(),
    Z=Z,
    meta=meta,
    target_idx=target_idx,
    context_cancer=TARGET_CANCER,
    sample_scope=str(notebook_cfg["evidence"]["sample_scope"]),
    top_m=int(notebook_cfg["evidence"]["top_m"]),
    explain_exemplars=int(notebook_cfg["evidence"]["explain_exemplars"]),
)

sample_scope = str(notebook_cfg["evidence"]["sample_scope"])
target_pair_mask = (
    ((meta["primary_gene"].astype(str) == TARGET_PRIMARY) & (meta["partner_gene"].astype(str) == TARGET_PARTNER))
    | ((meta["primary_gene"].astype(str) == TARGET_PARTNER) & (meta["partner_gene"].astype(str) == TARGET_PRIMARY))
)
if sample_scope == "all_cancer":
    eligible_count = int((~target_pair_mask).sum())
else:
    eligible_count = int(((meta["cancer"].astype(str) == TARGET_CANCER) & ~target_pair_mask).sum())
print("Evidence sample scope:", sample_scope)
print("Eligible non-target pairs:", eligible_count)
evidence_table.head()


## Joint Feature Ranking

Rank candidate features by combining rank-normalized local sensitivity and decoder geometry:

$$S_j=\frac{1}{2}\left(\operatorname{RankNorm}(|\dot z_j|)+\operatorname{RankNorm}(|c_j^*|)\right).$$

`RankNorm` maps each signal to its empirical rank in $[0,1]$ across the candidate feature set, so raw units of $\dot z_j$ and $c_j^*$ cannot dominate each other. Only ordering within each signal matters.


In [ ]:
feature_rank = rank_candidate_features(candidate_table, evidence_table)
selected_features = [int(feature) for feature in feature_rank.head(int(notebook_cfg["feature_selection"]["n_features_to_explain"]))["feature"]]

print("Selected features:", selected_features)
feature_rank.head(len(selected_features))

## Build Explainer Prompt


In [ ]:
preview_feature = selected_features[0]
preview_prompt = build_explainer_prompt(
    feature=preview_feature,
    feature_rank=feature_rank,
    explain_indices=explain_indices[preview_feature],
    Z=Z,
    y=y,
    meta=meta,
    target_idx=target_idx,
    target_cancer=TARGET_CANCER,
    sample_scope=str(notebook_cfg["evidence"]["sample_scope"]),
    prompt_dir=PROJECT_ROOT / "src" / "SAE" / "doc" / "prompts",
)
print(preview_prompt[:1800])


## Run The Feature Explainer


In [ ]:
%%time
client = AigcBestChatClient(config_path=prompt_api_config)
max_call_s = (
    int(client.settings.request_retries) * float(client.settings.request_timeout_s)
    + max(int(client.settings.request_retries) - 1, 0) * float(client.settings.retry_wait_s)
)

calls_per_feature = strategy_call_count(
    FEATURE_EXPLAINER_STRATEGY,
    self_refine_rounds=client.settings.self_refine_rounds,
    cove_num_questions=client.settings.cove_num_questions,
)
print(f"Planned LLM calls: {calls_per_feature * len(selected_features)} ({calls_per_feature} per feature)")
print(f"Worst-case configured wait per single API call: {max_call_s:.1f}s")

state_path = output_dir / "interpretation_state.json"
for stale_path in [
    output_dir / "llm_interpretation_test_details.json.gz",
    output_dir / "llm_interpretation_metrics.json",
]:
    if stale_path.exists():
        stale_path.unlink()
        print("Removed stale held-out artifact:", stale_path)
selected_feature_set = {int(feature) for feature in selected_features}
resume_completed = bool(notebook_cfg["runtime"]["resume_completed"])
interpretations = []
completed_features = set()

if resume_completed and state_path.exists():
    previous_state = json.loads(state_path.read_text(encoding="utf-8"))
    for row in previous_state["interpretations"]:
        feature = int(row["feature"])
        if feature in selected_feature_set and feature not in completed_features:
            row["feature"] = feature
            interpretations.append(row)
            completed_features.add(feature)
    print(f"Resume enabled: loaded {len(completed_features)} completed features from {state_path}")


def save_interpretation_artifacts() -> pd.DataFrame:
    summary = pd.DataFrame(interpretations)
    feature_rank.to_csv(output_dir / "feature_rank.csv", index=False)
    summary.to_csv(output_dir / "llm_interpretation_summary.csv", index=False)
    write_json(
        output_dir / "interpretation_state.json",
        {
            "target": {
                "primary_gene": TARGET_PRIMARY,
                "partner_gene": TARGET_PARTNER,
                "training_scope": TRAIN_SCOPE,
                "cancer": TARGET_CANCER,
                "target_idx": target_idx,
                "score": float(y[target_idx]),
            },
            "projection": {
                "neighbor_indices": local["neighbor_indices"].tolist(),
                "grad_ambient_norm": float(np.linalg.norm(local["grad_ambient"])),
                "x_tangent_norm": float(np.linalg.norm(state["x_tangent"])),
                "x_normal_norm": float(np.linalg.norm(state["x_normal"])),
                "decoder_projection": {
                    "alpha": decoder_projection["alpha"],
                    "nnz": decoder_projection["nnz"],
                    "cosine": decoder_projection["cosine"],
                    "rel_err": decoder_projection["rel_err"],
                },
            },
            "evidence": {
                "sample_scope": str(notebook_cfg["evidence"]["sample_scope"]),
                "top_m": int(notebook_cfg["evidence"]["top_m"]),
                "explain_exemplars": int(notebook_cfg["evidence"]["explain_exemplars"]),
            },
            "feature_rank": feature_rank.to_dict(orient="records"),
            "interpretations": interpretations,
        },
    )
    return summary


for position, feature in enumerate(selected_features, start=1):
    feature = int(feature)
    if feature in completed_features:
        print(f"[{position}/{len(selected_features)}] Feature {feature}: already completed, skipped")
        continue

    feature_row = feature_rank.loc[feature_rank["feature"] == feature].iloc[0]
    started = time.monotonic()
    explainer_prompt = build_explainer_prompt(
        feature=feature,
        feature_rank=feature_rank,
        explain_indices=explain_indices[feature],
        Z=Z,
        y=y,
        meta=meta,
        target_idx=target_idx,
        target_cancer=TARGET_CANCER,
        sample_scope=str(notebook_cfg["evidence"]["sample_scope"]),
        prompt_dir=client.settings.prompt_dir,
    )
    explainer_trace = run_llm_strategy(client, explainer_prompt, FEATURE_EXPLAINER_STRATEGY)
    explainer_text = final_strategy_text(explainer_trace)
    explainer_fields = extract_explainer_fields(explainer_text)
    interpretations.append(
        {
            "feature": feature,
            **explainer_fields,
            "target_relevance": target_relevance_note(feature_row),
            "explainer_strategy": FEATURE_EXPLAINER_STRATEGY,
            "evidence_sample_scope": str(notebook_cfg["evidence"]["sample_scope"]),
            "explainer_strategy_tokens": strategy_total_tokens(explainer_trace),
            "explainer_runtime_s": float(time.monotonic() - started),
        }
    )
    completed_features.add(feature)
    summary = save_interpretation_artifacts()
    print(f"[{position}/{len(selected_features)}] Feature {feature}: saved in {interpretations[-1]['explainer_runtime_s']:.1f}s")
    time.sleep(20)


## Explainer Output Summary


In [ ]:
summary = pd.DataFrame(interpretations)
summary_columns = [
    "feature",
    "hypothesis",
    "confidence",
    "target_relevance",
    "evidence_sample_scope",
    "explainer_strategy_tokens",
    "explainer_runtime_s",
]
if summary.empty:
    print("No completed feature explanations yet. Run the previous cell to populate interpretations.")
else:
    display(summary[[column for column in summary_columns if column in summary.columns]])
